<a href="https://colab.research.google.com/github/Sayandeepmaity/luminator/blob/main/gunshotdetecionandclassification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import librosa
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('dataset.csv')

In [ ]:
label_encoder_gun_type = LabelEncoder()
df['Gun Type'] = label_encoder_gun_type.fit_transform(df['Gun_Type'])

label_encoder_gun_model = LabelEncoder()
df['Gun Model'] = label_encoder_gun_model.fit_transform(df['Gun_Model'])

df['Firing Event'] = df['Firing Event'].astype('category').cat.codes
df['Source_Count'] = df['Source_Count'].astype(int)


In [ ]:
def compute_relative_positions(df):
    for i in range(1, 7):
        df[f'Distance_from_Primary_Mic_{i}'] = np.sqrt(
            (df[f'Mic({i}) X'] - df["Primary Mic_X"])**2 +
            (df[f'Mic({i}) Y'] - df["Primary Mic_Y"])**2 +
            (df[f'Mic({i}) Z'] - df["Primary Mic_Z"])**2
        )
    return df

df = compute_relative_positions(df)

def compute_tdoa(df):
    for i in range(2, 7):
        df[f'TDOA_{i}'] = df[f'Distance_from_Primary_Mic_1'] - df[f'Distance_from_Primary_Mic_{i}']
    return df

df = compute_tdoa(df)

In [ ]:
def extract_features(audio_file):
    y, sr = librosa.load(audio_file, sr=None)
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    spectral_centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
    return np.concatenate((np.mean(mfccs.T, axis=0), np.mean(spectral_centroid.T, axis=0)))

def augment_data(df):
    def apply_augmentation(row):
        audio_file = row['Audio File Path']
        y, sr = librosa.load(audio_file, sr=None)
        augmented_y = augment_audio(y, sr)
        augmented_file_path = audio_file.replace('.wav', '_augmented.wav')
        librosa.output.write_wav(augmented_file_path, augmented_y, sr)
        return augmented_file_path

    df['Augmented Audio File Path'] = df.apply(apply_augmentation, axis=1)
    return df

df = augment_data(df)

features = df[["Caliber", "Decibel Level (dB)", "Frequency Range (Hz)", "Distance_from_Primary_Mic_1",
               "Distance_from_Primary_Mic_2", "Distance_from_Primary_Mic_3", "Distance_from_Primary_Mic_4",
               "Distance_from_Primary_Mic_5", "Distance_from_Primary_Mic_6", "Temperature (Celsius)",
               "Humidity (%)", "Noise Level (dB)", "Microphone Sensitivity", "Echo Effect", "TDOA_1",
               "TDOA_2", "TDOA_3", "TDOA_4", "TDOA_5", "TDOA_6"]]

df["Source_X"] = df["Source_X"].astype(float)
df["Source_Y"] = df["Source_Y"].astype(float)
df["Source_Z"] = df["Source_Z"].astype(float)

target_sound = df[['Muzzle sound', 'Bowwave sound']]
target_gun = df[['Gun Type', 'Gun Model']]
target_firing_event = df[['Firing Event', 'Source_Count']]
target_location = df[['Source_X', 'Source_Y', 'Source_Z']]

scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)


In [ ]:
X_train, X_test, y_train_sound, y_test_sound, y_train_gun, y_test_gun, y_train_firing, y_test_firing, y_train_location, y_test_location = train_test_split(
    features_scaled, target_sound, target_gun, target_firing_event, target_location, test_size=0.2, random_state=42)


In [ ]:
def residual_block(x, units):
    shortcut = x
    x = layers.Dense(units, activation='relu')(x)
    x = layers.Dense(units)(x)
    x = layers.Add()([x, shortcut])
    x = layers.Activation('relu')(x)
    return x

input_features = layers.Input(shape=(X_train.shape[1],))
x = residual_block(input_features, 128)
x = residual_block(x, 64)

output_muzzle = layers.Dense(len(target_sound['Muzzle sound'].unique()), activation='softmax', name='muzzle_sound')(x)
output_shockwave = layers.Dense(len(target_sound['Bowwave sound'].unique()), activation='softmax', name='bowwave_sound')(x)
output_gun_type = layers.Dense(len(label_encoder_gun_type.classes_), activation='softmax', name='gun_type')(x)
output_gun_model = layers.Dense(len(label_encoder_gun_model.classes_), activation='softmax', name='gun_model')(x)
output_firing_event = layers.Dense(len(df['Firing Event'].unique()), activation='softmax', name='firing_event')(x)
output_source_count = layers.Dense(len(df['Source_Count'].unique()), activation='softmax', name='source_count')(x)
output_source_x = layers.Dense(1, name='source_x')(x)
output_source_y = layers.Dense(1, name='source_y')(x)
output_source_z = layers.Dense(1, name='source_z')(x)

model = Model(inputs=input_features,
              outputs=[output_muzzle, output_shockwave, output_gun_type, output_gun_model,
                       output_firing_event, output_source_count, output_source_x, output_source_y, output_source_z])

model.compile(optimizer='adam',
              loss={'muzzle_sound': 'sparse_categorical_crossentropy',
                    'bowwave_sound': 'sparse_categorical_crossentropy',
                    'gun_type': 'sparse_categorical_crossentropy',
                    'gun_model': 'sparse_categorical_crossentropy',
                    'firing_event': 'sparse_categorical_crossentropy',
                    'source_count': 'sparse_categorical_crossentropy',
                    'source_x': 'mse', 'source_y': 'mse', 'source_z': 'mse'},
              metrics={'muzzle_sound': 'accuracy',
                       'bowwave_sound': 'accuracy',
                       'gun_type': 'accuracy',
                       'gun_model': 'accuracy',
                       'firing_event': 'accuracy',
                       'source_count': 'accuracy',
                       'source_x': 'mse', 'source_y': 'mse', 'source_z': 'mse'})

In [ ]:
history = model.fit(X_train,
                    {'muzzle_sound': y_train_sound['Muzzle sound'],
                     'bowwave_sound': y_train_sound['Bowwave sound'],
                     'gun_type': y_train_gun['Gun Type'],
                     'gun_model': y_train_gun['Gun Model'],
                     'firing_event': y_train_firing['Firing Event'],
                     'source_count': y_train_firing['Source_Count'],
                     'source_x': y_train_location['Source_X'],
                     'source_y': y_train_location['Source_Y'],
                     'source_z': y_train_location['Source_Z']},
                    epochs=50, batch_size=32, validation_data=(X_test,
                     {'muzzle_sound': y_test_sound['Muzzle sound'],
                      'bowwave_sound': y_test_sound['Bowwave sound'],
                      'gun_type': y_test_gun['Gun Type'],
                      'gun_model': y_test_gun['Gun Model'],
                      'firing_event': y_test_firing['Firing Event'],
                      'source_count': y_test_firing['Source_Count'],
                      'source_x': y_test_location['Source_X'],
                      'source_y': y_test_location['Source_Y'],
                      'source_z': y_test_location['Source_Z']}))